My name is Matt. I've been programming in Python for the last three years, and consider myself intermediate. I would like to get a lot better. I care about the craft of software engineer and data engineering, and I want you to be my tutor in helping to understand both the specifics, but especially the fundamentals, more deeply. I'm working on a personal project to build an end-to-end data pipeline that pulls live energy data from the ENTSO-E API.

Personal Project Summary: Energy Grid Data Pipeline

What it is: A self-directed portfolio project to build a complete data pipeline, end to end, using industry-standard data engineering tools and practices.

What it does: Pulls live electricity grid data (generation, demand, prices) from ENTSO-E, the EU's official energy transparency API, on a recurring schedule. Loads it into a database, transforms it into clean, tested, analysis-ready tables, and eventually serves it via a small dashboard.

Why I'm doing it: To build and demonstrate skills beyond my current role — specifically: pipeline orchestration, SQL-based transformation (dbt), cloud infrastructure (AWS), containerization, and CI/CD. These are core data engineering skills I don't currently use day to day.

Tech stack: Python, Postgres, dbt, Docker, an orchestrator (Prefect, Dagster or Airflow), and AWS (S3, RDS/Lambda or ECS) once the local version is working.

Approach: Built in stages — a minimal working version first (API → database, running manually), then scheduling, then transformations, then orchestration, then containerization, then cloud deployment, then CI/CD. Each stage is a working, demoable system on its own.

Status: Early build phase — currently writing the first data extraction script.

In [6]:
import httpx
import os
from pathlib import Path
api_key = os.getenv('ENTSOE')

In [3]:
params = {
    "securityToken": api_key,
    "documentType": "A75",
    "processType": "A16",
    "in_Domain": "10Y1001A1001A82H", # DE-LU
    "periodStart": "202608152200",
    "periodEnd": "202608162200",
}

timeout = httpx.Timeout(connect=10.0, read=180.0, write=10.0, pool=10.0)

r = httpx.get(
    "https://web-api.tp.entsoe.eu/api",
    params=params,
    timeout=timeout,
)

In [4]:
r.status_code

200

In [5]:
r.content

b'<?xml version="1.0" encoding="utf-8"?>\n<GL_MarketDocument xmlns="urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0">\n  <mRID>5aa04bf9cf9247b68fe5ba34ffe965fe</mRID>\n  <revisionNumber>1</revisionNumber>\n  <type>A75</type>\n  <process.processType>A16</process.processType>\n  <sender_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</sender_MarketParticipant.mRID>\n  <sender_MarketParticipant.marketRole.type>A32</sender_MarketParticipant.marketRole.type>\n  <receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</receiver_MarketParticipant.mRID>\n  <receiver_MarketParticipant.marketRole.type>A33</receiver_MarketParticipant.marketRole.type>\n  <createdDateTime>2026-09-09T15:45:22Z</createdDateTime>\n  <time_Period.timeInterval>\n    <start>2026-08-15T22:00Z</start>\n    <end>2026-08-16T22:00Z</end>\n  </time_Period.timeInterval>\n      <TimeSeries>\n        <mRID>1</mRID>\n        <businessType>A01</businessType>\n        <objectAggregation>A08</objectA